In [9]:
import os
num_cpu_threads = '1'
os.environ['OMP_NUM_THREADS']=num_cpu_threads
os.environ['MKL_NUM_THREADS']=num_cpu_threads
os.environ['OPENBLAS_NUM_THREADS']=num_cpu_threads
os.environ['NUMEXPR_NUM_THREADS']=num_cpu_threads

import time
import torch
import shutil
import warnings
import logging
import numpy as np
from xtb.ase.calculator import XTB
from mace.calculators import MACECalculator
from ase.utils.forcecurve import fit_images, plotfromfile
from ase.io.trajectory import TrajectoryReader
from ase.calculators.mixing import SumCalculator
from sella import Sella
from sella import IRC
from x3dase.x3d import X3D
from ase.io import read
from ase.vibrations import Vibrations
from ase.mep import DimerControl, MinModeAtoms, MinModeTranslate
import matplotlib.pyplot as plt
from ase.io import read, write, animation
from ase.optimize.bfgs import BFGS
from natsort import ns, natsorted
import configparser
import traceback

#Using cuda with jax results in more VRAM usage and worse performance, so we'll make sure jax runs on CPU instead.
os.environ['JAX_PLATFORMS'] = 'cpu'

In [12]:
def ts_irc_pipeline(inputxyz, output_path, rxn_output_path, calculator,
                    redine_f_max=0.01, irc_f_max=0.01, steps=1000, retry=2):
    """
    The main process includes transition state optimization, IRC path optimization, and two-end minima optimization.
    Automatically records logs and critical path parameters, and supports a retry mechanism.
    """

    rxn_name = os.path.splitext(os.path.basename(inputxyz))[0]

    # Setup logging
    logfile_path = os.path.join(rxn_output_path, f'{rxn_name}.log')
    logging.basicConfig(filename=logfile_path,
                        filemode='w',
                        level=logging.INFO,
                        format='%(asctime)s - %(levelname)s - %(message)s')
    log = logging.getLogger()

    # Summary log header
    all_log_file = os.path.join(output_path, 'all_log.txt')
    if not os.path.exists(all_log_file):
        with open(all_log_file, 'w') as f:
            f.write('name\tref_time\trefine_steps\tirc_time\tirc_steps\tEf\tEr\tflag\n')

    for attempt in range(1, retry + 1):
        try:
            # Step 1: TS Optimization
            log.info(f'[Step 1] TS optimization for {rxn_name}')
            atoms = read(inputxyz)
            atoms.calc = calculator
            atoms.info = {'charge': 0, 'spin': 1}

            ts_traj_path = os.path.join(rxn_output_path, f'{rxn_name}_ts.traj')
            ts_final_xyz = os.path.join(rxn_output_path, f'{rxn_name}_ts.xyz')

            t0 = time.time()
            dyn = Sella(atoms, trajectory=ts_traj_path, eta=2e-2, gamma=0.0001, delta0=0.02)
            dyn.run(fmax=redine_f_max, steps=steps)
            t1 = time.time()
            ref_time = round(t1 - t0, 2)

            traj = TrajectoryReader(ts_traj_path)
            refine_steps = len(traj) - 1
            ts_final = traj[-1]
            write(ts_final_xyz, ts_final)

            log.info(f'Finished TS optimization in {ref_time}s, steps: {refine_steps}')

            # Step 2: Frequency Check
            vib_dir = os.path.join(rxn_output_path, 'vib')
            if os.path.exists(vib_dir):
                shutil.rmtree(vib_dir)
            ts_final.info = {'charge': 0, 'spin': 1}
            ts_final.calc = calculator
            vib = Vibrations(ts_final, name=vib_dir)
            vib.run()
            freqs = vib.get_frequencies()
            imaginary_count = np.sum(np.imag(freqs) != 0)
            log.info(f'Imaginary frequencies: {imaginary_count}')
            vib.summary(log=os.path.join(rxn_output_path, f'{rxn_name}_vibration.log'))

            print(f' TS optimization successful for {rxn_name}!')

            if imaginary_count > 6:
                log.warning(f'Too many imaginary modes (>6). Retrying...')
                continue

            # Step 3: IRC
            irc_traj_path = os.path.join(rxn_output_path, f'{rxn_name}_irc.traj')
            ts_final.calc = calculator
            log.info(f'[Step 3] IRC calculation for {rxn_name}')
            irc = IRC(ts_final, trajectory=irc_traj_path, dx=0.1, eta=1e-4, gamma=0.1)
            t2 = time.time()
            irc.run(fmax=irc_f_max, steps=steps, direction='forward')
            split = len(TrajectoryReader(irc_traj_path))
            irc.run(fmax=irc_f_max, steps=steps, direction='reverse')
            t3 = time.time()
            irc_time = round(t3 - t2, 2)
            log.info(f'IRC completed in {irc_time}s. Split index: {split}')

            irc_traj = TrajectoryReader(irc_traj_path)
            irc_steps = len(irc_traj)

            # Step 4: Visualize IRC
            irc_all = [irc_traj[i] for i in range(split - 1, -1, -1)]
            irc_all.extend(irc_traj[split:])
            try:
                fit = fit_images(irc_all)
                fit.plot()
                plt.savefig(os.path.join(rxn_output_path, f'{rxn_name}_irc.png'), dpi=300)
                animation.write_animation(os.path.join(rxn_output_path, f'{rxn_name}_irc.gif'),
                                          images=irc_all, writer='pillow', interval=50)
                plt.close()
            except Exception as e:
                log.warning('IRC visualization failed.')

            # Step 5: Optimize IRC endpoints
            atoms1, atoms2 = irc_all[0], irc_all[-1]
            for i, atoms in enumerate([atoms1, atoms2], 1):
                atoms.calc = calculator
                opt = BFGS(atoms, trajectory=os.path.join(rxn_output_path, f'{rxn_name}_opt{i}.traj'))
                opt.run(fmax=0.01, steps=steps)

            opt1 = TrajectoryReader(os.path.join(rxn_output_path, f'{rxn_name}_opt1.traj'))[-1]
            opt2 = TrajectoryReader(os.path.join(rxn_output_path, f'{rxn_name}_opt2.traj'))[-1]
            e1 = opt1.get_potential_energy()
            e2 = opt2.get_potential_energy()

            Ef = max([a.get_potential_energy() for a in irc_all]) - max(e1, e2)
            Er = max([a.get_potential_energy() for a in irc_all]) - min(e1, e2)

            # Save lower energy product/reactant
            try:
                if e1 > e2:
                    write(os.path.join(rxn_output_path, f'{rxn_name}_r.xyz'), opt1)
                    write(os.path.join(rxn_output_path, f'{rxn_name}_p.xyz'), opt2)
            except Exception as e:
                log.warning('Failure of structural minimization.')

            #Step 6: Write concise logs
            with open(os.path.join(rxn_output_path, 'run.log'), 'w') as f:
                f.write(f'ref_time: {ref_time}\n')
                f.write(f'refine_steps: {refine_steps}\n')
                f.write(f'irc_time: {irc_time}\n')
                f.write(f'irc_steps: {irc_steps}\n')
                f.write(f'Ef: {Ef:.6f} eV\n')
                f.write(f'Er: {Er:.6f} eV\n')
                f.write(f'irc_split_flag: {split}\n')

            with open(all_log_file, 'a') as f:
                f.write(f'{rxn_name}\t{ref_time}\t{refine_steps}\t{irc_time}\t{irc_steps}\t{Ef:.6f}\t{Er:.6f}\t{split}\n')

            log.info(f'>> {rxn_name} successfully finished.')
            return  # Success, exit loop

        except Exception as e:
            log.error(f'Exception during attempt {attempt} for {rxn_name}: {str(e)}')
            traceback.print_exc()
            if attempt < retry:
                log.warning(f'Retrying {rxn_name} (attempt {attempt + 1}/{retry})...')
                time.sleep(2)
            else:
                log.error(f'{rxn_name} failed after {retry} attempts.')
                return  # Final failure

In [13]:
if __name__ == '__main__':

    # Model name and input file path
    current_dir = os.getcwd()
    input_path = f'{current_dir}/inputs'
    output_path = f'{current_dir}/outputs'
    rxn_names = [n for n in os.listdir(input_path) if n.endswith('.xyz')]
    rxn_names = natsorted(rxn_names, alg=ns.PATH)
    print(f'number of jobs: {len(rxn_names)}.')

    # Model path
    model_path = os.path.join(current_dir, "../../models/MACE_deltaL/MACE_deltaL.model")
    model_path = os.path.abspath(model_path)

    # Define the mixed calculator of MACE-delta model
    calc1 = MACECalculator(model_paths=model_path, device='cuda')
    calc2 = XTB(method='GFN2-xTB')
    calculator = SumCalculator([calc1, calc2])

    # Batch execution ts opt and irc calculations
    for rx in rxn_names:
        rxn_name = rx.split('.')[0]
        # work folder
        rxn_output_path = os.path.join(output_path, rxn_name)
        if not os.path.exists(rxn_output_path):
            os.makedirs(rxn_output_path)
        else:
            print(f'{rxn_name} already exists, skip.')
            continue

        input_xyz = os.path.join(input_path, rx)
        ts_irc_pipeline(input_xyz, output_path, rxn_output_path, calculator, redine_f_max=0.0025, irc_f_max=0.005, steps=1000)

number of jobs: 2.
Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


/home/tangkun/.local/lib/python3.10/site-packages/ase/optimize/optimize.py:372: FutureWarning: force_consistent keyword is deprecated and will be ignored.  This will raise an error in future versions of ASE.
  warnings.warn(


     Step     Time          Energy         fmax         cmax       rtrust          rho
Sella   0 16:52:11   -11343.423866       1.8294       0.0000       0.0200       1.0000
Sella   1 16:52:12   -11343.475763       0.7150       0.0000       0.0230       0.9728
Sella   2 16:52:12   -11343.495744       0.2584       0.0000       0.0264       0.9752
Sella   3 16:52:12   -11343.498917       0.0327       0.0000       0.0264       1.1141
Sella   4 16:52:12   -11343.499002       0.0116       0.0000       0.0264       2.7642
Sella   5 16:52:12   -11343.498232       0.0091       0.0000       0.0200     -57.4165
Sella   6 16:52:12   -11343.499008       0.0086       0.0000       0.0200      43.8052
Sella   7 16:52:12   -11343.498828       0.0048       0.0000       0.0200     -50.2119
Sella   8 16:52:12   -11343.498751       0.0038       0.0000       0.0200     -65.5004
Sella   9 16:52:12   -11343.498762       0.0017       0.0000       0.0200      11.3579
 TS optimization successful for ts_1_1_NO2-

/opt/miniconda3/envs/rmlp/lib/python3.10/site-packages/sella/peswrapper.py:325: RuntimeWarning: invalid value encountered in scalar divide
  ratio = df_actual / df_pred


     Step     Time          Energy          fmax
IRC:    0 16:52:17   -11343.498762        0.001740
IRC:    1 16:52:17   -11343.565886        1.272933
IRC:    2 16:52:17   -11343.750147        2.226143
IRC:    3 16:52:17   -11344.004920        2.567133
IRC:    4 16:52:17   -11344.258242        2.186776
IRC:    5 16:52:18   -11344.454091        1.985482
IRC:    6 16:52:18   -11344.587025        1.611391
IRC:    7 16:52:18   -11344.676687        1.323264
IRC:    8 16:52:18   -11344.738340        1.113089
IRC:    9 16:52:18   -11344.782834        0.886142
IRC:   10 16:52:18   -11344.816259        0.683427
IRC:   11 16:52:18   -11344.841919        0.513760
IRC:   12 16:52:19   -11344.860098        0.382306
IRC:   13 16:52:19   -11344.872927        0.282059
IRC:   14 16:52:19   -11344.882336        0.209594
IRC:   15 16:52:19   -11344.889328        0.159826
IRC:   16 16:52:19   -11344.894558        0.122332
IRC:   17 16:52:19   -11344.899508        0.093450
IRC:   18 16:52:19   -11344.90298

/home/tangkun/.local/lib/python3.10/site-packages/ase/optimize/optimize.py:372: FutureWarning: force_consistent keyword is deprecated and will be ignored.  This will raise an error in future versions of ASE.
  warnings.warn(


     Step     Time          Energy         fmax         cmax       rtrust          rho
Sella   0 16:52:24   -16009.630098       0.9834       0.0000       0.0200       1.0000
Sella   1 16:52:25   -16009.668153       0.4914       0.0000       0.0230       0.9994
Sella   2 16:52:25   -16009.694119       0.1988       0.0000       0.0264       0.9976
Sella   3 16:52:25   -16009.703816       0.1090       0.0000       0.0304       1.0286
Sella   4 16:52:25   -16009.708494       0.0519       0.0000       0.0304       1.1682
Sella   5 16:52:26   -16009.708985       0.0321       0.0000       0.0304       0.6011
Sella   6 16:52:26   -16009.709492       0.0566       0.0000       0.0304       2.3672
Sella   7 16:52:26   -16009.710031       0.0259       0.0000       0.0304       2.6228
Sella   8 16:52:26   -16009.710262       0.0293       0.0000       0.0304       1.9435
Sella   9 16:52:26   -16009.710193       0.0238       0.0000       0.0200      -0.5290
Sella  10 16:52:26   -16009.710226       0.

/opt/miniconda3/envs/rmlp/lib/python3.10/site-packages/sella/peswrapper.py:325: RuntimeWarning: invalid value encountered in scalar divide
  ratio = df_actual / df_pred


     Step     Time          Energy          fmax
IRC:    0 16:52:34   -16009.711733        0.001750
IRC:    1 16:52:34   -16009.744924        0.914632
IRC:    2 16:52:35   -16009.825762        1.616150
IRC:    3 16:52:35   -16009.910719        1.953980
IRC:    4 16:52:35   -16009.992848        2.052394
IRC:    5 16:52:35   -16010.074485        2.048287
IRC:    6 16:52:35   -16010.156158        2.000659
IRC:    7 16:52:36   -16010.236685        1.930448
IRC:    8 16:52:36   -16010.318045        1.848791
IRC:    9 16:52:36   -16010.397338        1.887468
IRC:   10 16:52:36   -16010.476333        1.940656
IRC:   11 16:52:36   -16010.554892        1.975609
IRC:   12 16:52:36   -16010.631798        1.991251
IRC:   13 16:52:36   -16010.706695        1.986252
IRC:   14 16:52:36   -16010.780293        1.958287
IRC:   15 16:52:36   -16010.851166        1.906600
IRC:   16 16:52:37   -16010.918939        1.829230
IRC:   17 16:52:37   -16010.983021        1.728272
IRC:   18 16:52:37   -16011.04310